# Data Asset Intelligence Graph — Seed Data

This notebook loads a realistic synthetic dataset into your Neo4j AuraDB instance.

**What it creates:**
- 7 Data Vendors
- 14 Datasets with contract metadata
- 8 Feature Pipelines
- 22 Features with Shapley values
- 6 Models (live + research)
- 4 Strategies with AUM
- PnL records across 4 periods
- Substitutability and correlation relationships

**Estimated load time:** < 30 seconds

## 0. Install dependencies

In [ ]:
%pip install neo4j pandas --quiet

## 1. Configure connection

Set your AuraDB credentials below. You can find them in the **Aura Console** under your instance's **Connection details**.

In [ ]:
NEO4J_URI      = "neo4j+s://<your-instance>.databases.neo4j.io"  # Replace
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "<your-password>"  # Replace

from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("✅ Connected to Neo4j")

## 2. Clear existing data (optional)

> ⚠️ This deletes **all** nodes and relationships in the database. Skip if you want to preserve existing data.

In [ ]:
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("🗑️  Database cleared")

## 3. Create constraints and indexes

In [ ]:
constraints = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:DataVendor)      REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Dataset)         REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:FeaturePipeline) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Feature)         REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Model)           REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Strategy)        REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:PnL)             REQUIRE n.id IS UNIQUE",
]

with driver.session() as session:
    for c in constraints:
        session.run(c)
print("✅ Constraints created")

## 4. Load Data Vendors

In [ ]:
vendors = [
    {"id": "v1", "name": "OrbitalView",      "category": "Alternative Data",  "hq_country": "USA",     "tier": "Tier1"},
    {"id": "v2", "name": "TransactIQ",        "category": "Alternative Data",  "hq_country": "USA",     "tier": "Tier1"},
    {"id": "v3", "name": "SentimentLab",      "category": "Alternative Data",  "hq_country": "UK",      "tier": "Tier2"},
    {"id": "v4", "name": "MacroMatrix",        "category": "Macro & Economic",  "hq_country": "Germany", "tier": "Tier1"},
    {"id": "v5", "name": "FlowEdge",           "category": "Market Microstructure", "hq_country": "USA", "tier": "Tier2"},
    {"id": "v6", "name": "GreenSignal",        "category": "ESG & Alternative",  "hq_country": "France", "tier": "Tier2"},
    {"id": "v7", "name": "TalentPulse",        "category": "Alternative Data",  "hq_country": "USA",     "tier": "Tier3"},
]

query = """
UNWIND $vendors AS v
MERGE (n:DataVendor {id: v.id})
SET n += v
"""
with driver.session() as session:
    session.run(query, vendors=vendors)
print(f"✅ {len(vendors)} vendors loaded")

## 5. Load Datasets

In [ ]:
datasets = [
    # OrbitalView
    {"id": "d1",  "name": "Parking Lot Occupancy - US Retail",   "vendor_id": "v1", "category": "Satellite",
     "asset_class_coverage": "US Equities",   "annual_license_cost": 480000, "ingestion_cost": 35000,
     "signal_half_life_days": 120, "mnpi_risk_score": 0.3, "coverage_start_year": 2016,
     "refresh_frequency": "Weekly",  "active": True},
    {"id": "d2",  "name": "Industrial Activity Index - Global",   "vendor_id": "v1", "category": "Satellite",
     "asset_class_coverage": "Global Equities", "annual_license_cost": 620000, "ingestion_cost": 40000,
     "signal_half_life_days": 90,  "mnpi_risk_score": 0.2, "coverage_start_year": 2017,
     "refresh_frequency": "Monthly", "active": True},
    # TransactIQ
    {"id": "d3",  "name": "Consumer Credit Card Spend - US",     "vendor_id": "v2", "category": "Credit Card Transactions",
     "asset_class_coverage": "US Equities",   "annual_license_cost": 750000, "ingestion_cost": 55000,
     "signal_half_life_days": 60,  "mnpi_risk_score": 0.4, "coverage_start_year": 2015,
     "refresh_frequency": "Daily",   "active": True},
    {"id": "d4",  "name": "Consumer Credit Card Spend - Europe", "vendor_id": "v2", "category": "Credit Card Transactions",
     "asset_class_coverage": "European Equities", "annual_license_cost": 520000, "ingestion_cost": 45000,
     "signal_half_life_days": 75,  "mnpi_risk_score": 0.4, "coverage_start_year": 2018,
     "refresh_frequency": "Daily",   "active": True},
    # SentimentLab
    {"id": "d5",  "name": "Earnings Call NLP Sentiment",         "vendor_id": "v3", "category": "NLP / Sentiment",
     "asset_class_coverage": "Global Equities", "annual_license_cost": 195000, "ingestion_cost": 20000,
     "signal_half_life_days": 30,  "mnpi_risk_score": 0.6, "coverage_start_year": 2014,
     "refresh_frequency": "Daily",   "active": True},
    {"id": "d6",  "name": "News & Social Media Sentiment",       "vendor_id": "v3", "category": "NLP / Sentiment",
     "asset_class_coverage": "Global Equities", "annual_license_cost": 145000, "ingestion_cost": 15000,
     "signal_half_life_days": 14,  "mnpi_risk_score": 0.1, "coverage_start_year": 2017,
     "refresh_frequency": "Real-time", "active": True},
    # MacroMatrix
    {"id": "d7",  "name": "Nowcast GDP & Inflation - G20",       "vendor_id": "v4", "category": "Macro / Economic",
     "asset_class_coverage": "Multi-Asset",    "annual_license_cost": 310000, "ingestion_cost": 25000,
     "signal_half_life_days": 180, "mnpi_risk_score": 0.05,"coverage_start_year": 2010,
     "refresh_frequency": "Monthly", "active": True},
    {"id": "d8",  "name": "Central Bank Communication Index",    "vendor_id": "v4", "category": "Macro / Economic",
     "asset_class_coverage": "Fixed Income",   "annual_license_cost": 220000, "ingestion_cost": 18000,
     "signal_half_life_days": 45,  "mnpi_risk_score": 0.1, "coverage_start_year": 2012,
     "refresh_frequency": "Daily",   "active": True},
    # FlowEdge
    {"id": "d9",  "name": "Options Flow & Unusual Activity",     "vendor_id": "v5", "category": "Market Microstructure",
     "asset_class_coverage": "US Equities",   "annual_license_cost": 380000, "ingestion_cost": 30000,
     "signal_half_life_days": 10,  "mnpi_risk_score": 0.7, "coverage_start_year": 2019,
     "refresh_frequency": "Real-time", "active": True},
    {"id": "d10", "name": "Dark Pool & Off-Exchange Flow",       "vendor_id": "v5", "category": "Market Microstructure",
     "asset_class_coverage": "US Equities",   "annual_license_cost": 290000, "ingestion_cost": 28000,
     "signal_half_life_days": 5,   "mnpi_risk_score": 0.6, "coverage_start_year": 2020,
     "refresh_frequency": "Daily",   "active": True},
    # GreenSignal
    {"id": "d11", "name": "Corporate ESG Controversy Tracker",   "vendor_id": "v6", "category": "ESG",
     "asset_class_coverage": "Global Equities", "annual_license_cost": 160000, "ingestion_cost": 12000,
     "signal_half_life_days": 200, "mnpi_risk_score": 0.05,"coverage_start_year": 2018,
     "refresh_frequency": "Weekly",  "active": True},
    {"id": "d12", "name": "Supply Chain Emissions Footprint",    "vendor_id": "v6", "category": "ESG",
     "asset_class_coverage": "Global Equities", "annual_license_cost": 95000,  "ingestion_cost": 10000,
     "signal_half_life_days": 365, "mnpi_risk_score": 0.05,"coverage_start_year": 2020,
     "refresh_frequency": "Monthly", "active": False},  # Inactive — under review
    # TalentPulse
    {"id": "d13", "name": "Job Posting Velocity by Sector",      "vendor_id": "v7", "category": "Web Scraping",
     "asset_class_coverage": "US Equities",   "annual_license_cost": 85000,  "ingestion_cost": 22000,
     "signal_half_life_days": 45,  "mnpi_risk_score": 0.1, "coverage_start_year": 2019,
     "refresh_frequency": "Weekly",  "active": True},
    {"id": "d14", "name": "Executive LinkedIn Activity Index",   "vendor_id": "v7", "category": "Web Scraping",
     "asset_class_coverage": "US Equities",   "annual_license_cost": 60000,  "ingestion_cost": 18000,
     "signal_half_life_days": 20,  "mnpi_risk_score": 0.3, "coverage_start_year": 2021,
     "refresh_frequency": "Weekly",  "active": True},
]

contract_meta = [
    {"dataset_id": "d1",  "vendor_id": "v1", "contract_start": "2021-01-01", "renewal_date": "2025-06-30", "notice_period_days": 90,  "annual_fee_usd": 480000, "exclusivity": False, "auto_renew": True},
    {"dataset_id": "d2",  "vendor_id": "v1", "contract_start": "2022-03-01", "renewal_date": "2025-03-01", "notice_period_days": 60,  "annual_fee_usd": 620000, "exclusivity": True,  "auto_renew": False},
    {"dataset_id": "d3",  "vendor_id": "v2", "contract_start": "2020-01-01", "renewal_date": "2025-01-01", "notice_period_days": 90,  "annual_fee_usd": 750000, "exclusivity": False, "auto_renew": True},
    {"dataset_id": "d4",  "vendor_id": "v2", "contract_start": "2022-06-01", "renewal_date": "2025-06-01", "notice_period_days": 90,  "annual_fee_usd": 520000, "exclusivity": False, "auto_renew": False},
    {"dataset_id": "d5",  "vendor_id": "v3", "contract_start": "2021-09-01", "renewal_date": "2025-09-01", "notice_period_days": 30,  "annual_fee_usd": 195000, "exclusivity": False, "auto_renew": True},
    {"dataset_id": "d6",  "vendor_id": "v3", "contract_start": "2023-01-01", "renewal_date": "2025-12-31", "notice_period_days": 30,  "annual_fee_usd": 145000, "exclusivity": False, "auto_renew": True},
    {"dataset_id": "d7",  "vendor_id": "v4", "contract_start": "2019-01-01", "renewal_date": "2026-01-01", "notice_period_days": 120, "annual_fee_usd": 310000, "exclusivity": False, "auto_renew": True},
    {"dataset_id": "d8",  "vendor_id": "v4", "contract_start": "2020-06-01", "renewal_date": "2025-06-01", "notice_period_days": 60,  "annual_fee_usd": 220000, "exclusivity": False, "auto_renew": False},
    {"dataset_id": "d9",  "vendor_id": "v5", "contract_start": "2022-01-01", "renewal_date": "2025-04-01", "notice_period_days": 30,  "annual_fee_usd": 380000, "exclusivity": False, "auto_renew": False},
    {"dataset_id": "d10", "vendor_id": "v5", "contract_start": "2023-01-01", "renewal_date": "2025-07-01", "notice_period_days": 30,  "annual_fee_usd": 290000, "exclusivity": False, "auto_renew": True},
    {"dataset_id": "d11", "vendor_id": "v6", "contract_start": "2022-01-01", "renewal_date": "2026-01-01", "notice_period_days": 60,  "annual_fee_usd": 160000, "exclusivity": False, "auto_renew": True},
    {"dataset_id": "d12", "vendor_id": "v6", "contract_start": "2022-01-01", "renewal_date": "2025-05-01", "notice_period_days": 60,  "annual_fee_usd": 95000,  "exclusivity": False, "auto_renew": False},
    {"dataset_id": "d13", "vendor_id": "v7", "contract_start": "2023-04-01", "renewal_date": "2025-04-01", "notice_period_days": 30,  "annual_fee_usd": 85000,  "exclusivity": False, "auto_renew": False},
    {"dataset_id": "d14", "vendor_id": "v7", "contract_start": "2023-07-01", "renewal_date": "2025-07-01", "notice_period_days": 30,  "annual_fee_usd": 60000,  "exclusivity": False, "auto_renew": False},
]

# Create Dataset nodes
q_ds = """
UNWIND $datasets AS d
MERGE (n:Dataset {id: d.id})
SET n += d
"""

# Create PROVIDES relationships
q_prov = """
UNWIND $contracts AS c
MATCH (v:DataVendor {id: c.vendor_id})
MATCH (d:Dataset    {id: c.dataset_id})
MERGE (v)-[r:PROVIDES]->(d)
SET r += {contract_start:     c.contract_start,
          renewal_date:       c.renewal_date,
          notice_period_days: c.notice_period_days,
          annual_fee_usd:     c.annual_fee_usd,
          exclusivity:        c.exclusivity,
          auto_renew:         c.auto_renew}
"""

with driver.session() as session:
    session.run(q_ds,   datasets=datasets)
    session.run(q_prov, contracts=contract_meta)
print(f"✅ {len(datasets)} datasets and {len(contract_meta)} PROVIDES relationships loaded")

## 6. Load Feature Pipelines & Features

In [ ]:
pipelines = [
    {"id": "fp1", "name": "Retail Footfall Signal",        "team_owner": "Alt Data Research",    "compute_cost_annual": 18000, "language": "Python"},
    {"id": "fp2", "name": "Global Activity Composite",     "team_owner": "Macro Quant",           "compute_cost_annual": 22000, "language": "Python"},
    {"id": "fp3", "name": "Consumer Spend Analytics",      "team_owner": "Equity Alpha",          "compute_cost_annual": 32000, "language": "Python"},
    {"id": "fp4", "name": "NLP Sentiment Engine",          "team_owner": "ML Platform",           "compute_cost_annual": 45000, "language": "Python"},
    {"id": "fp5", "name": "Macro Factor Builder",          "team_owner": "Macro Quant",           "compute_cost_annual": 15000, "language": "Python"},
    {"id": "fp6", "name": "Microstructure Flow Signals",   "team_owner": "Execution Research",   "compute_cost_annual": 28000, "language": "C++"},
    {"id": "fp7", "name": "ESG Risk Overlay",              "team_owner": "Risk & Compliance",    "compute_cost_annual": 9000,  "language": "Python"},
    {"id": "fp8", "name": "Workforce Intelligence Signal", "team_owner": "Alt Data Research",    "compute_cost_annual": 12000, "language": "Python"},
]

pipeline_dataset_links = [
    # fp1 — Retail Footfall Signal
    {"fp": "fp1", "ds": "d1",  "primary_input": True},
    # fp2 — Global Activity Composite
    {"fp": "fp2", "ds": "d2",  "primary_input": True},
    {"fp": "fp2", "ds": "d7",  "primary_input": False},
    # fp3 — Consumer Spend Analytics
    {"fp": "fp3", "ds": "d3",  "primary_input": True},
    {"fp": "fp3", "ds": "d4",  "primary_input": True},
    # fp4 — NLP Sentiment Engine
    {"fp": "fp4", "ds": "d5",  "primary_input": True},
    {"fp": "fp4", "ds": "d6",  "primary_input": False},
    # fp5 — Macro Factor Builder
    {"fp": "fp5", "ds": "d7",  "primary_input": True},
    {"fp": "fp5", "ds": "d8",  "primary_input": True},
    # fp6 — Microstructure Flow Signals
    {"fp": "fp6", "ds": "d9",  "primary_input": True},
    {"fp": "fp6", "ds": "d10", "primary_input": False},
    # fp7 — ESG Risk Overlay
    {"fp": "fp7", "ds": "d11", "primary_input": True},
    {"fp": "fp7", "ds": "d12", "primary_input": False},
    # fp8 — Workforce Intelligence Signal
    {"fp": "fp8", "ds": "d13", "primary_input": True},
    {"fp": "fp8", "ds": "d14", "primary_input": False},
]

q_fp = """
UNWIND $pipelines AS p
MERGE (n:FeaturePipeline {id: p.id})
SET n += p
"""

q_feeds = """
UNWIND $links AS l
MATCH (d:Dataset        {id: l.ds})
MATCH (p:FeaturePipeline {id: l.fp})
MERGE (d)-[r:FEEDS]->(p)
SET r.primary_input = l.primary_input
"""

with driver.session() as session:
    session.run(q_fp,    pipelines=pipelines)
    session.run(q_feeds, links=pipeline_dataset_links)
print(f"✅ {len(pipelines)} pipelines and {len(pipeline_dataset_links)} FEEDS relationships loaded")

In [ ]:
features = [
    # fp1 → Retail Footfall Signal
    {"id": "f1",  "name": "Retail Footfall Momentum (30d)",          "pipeline": "fp1", "factor_type": "Alternative",  "coverage_universe": "US Equities",       "lookback_days": 30,  "is_proprietary": False, "transformation": "Rolling z-score"},
    {"id": "f2",  "name": "Retail Footfall Trend Reversal",          "pipeline": "fp1", "factor_type": "Alternative",  "coverage_universe": "US Equities",       "lookback_days": 90,  "is_proprietary": False, "transformation": "Mean reversion signal"},
    # fp2 → Global Activity Composite
    {"id": "f3",  "name": "Industrial Activity YoY Change",          "pipeline": "fp2", "factor_type": "Macro",         "coverage_universe": "Global Equities",   "lookback_days": 365, "is_proprietary": False, "transformation": "Year-on-year delta"},
    {"id": "f4",  "name": "EM vs DM Activity Divergence",            "pipeline": "fp2", "factor_type": "Macro",         "coverage_universe": "Global Equities",   "lookback_days": 180, "is_proprietary": True,  "transformation": "Cross-sectional spread"},
    # fp3 → Consumer Spend Analytics
    {"id": "f5",  "name": "Consumer Spend Growth (60d)",             "pipeline": "fp3", "factor_type": "Alternative",  "coverage_universe": "US Equities",       "lookback_days": 60,  "is_proprietary": False, "transformation": "Sector-relative growth"},
    {"id": "f6",  "name": "Discretionary vs Staples Spend Ratio",   "pipeline": "fp3", "factor_type": "Alternative",  "coverage_universe": "US Equities",       "lookback_days": 30,  "is_proprietary": True,  "transformation": "Ratio normalization"},
    {"id": "f7",  "name": "European Consumer Recovery Index",        "pipeline": "fp3", "factor_type": "Alternative",  "coverage_universe": "European Equities", "lookback_days": 90,  "is_proprietary": False, "transformation": "Composite index"},
    # fp4 → NLP Sentiment Engine
    {"id": "f8",  "name": "Earnings Call Tone Score",                "pipeline": "fp4", "factor_type": "Sentiment",    "coverage_universe": "Global Equities",   "lookback_days": 90,  "is_proprietary": False, "transformation": "BERT fine-tuned classifier"},
    {"id": "f9",  "name": "News Sentiment Momentum",                 "pipeline": "fp4", "factor_type": "Sentiment",    "coverage_universe": "Global Equities",   "lookback_days": 14,  "is_proprietary": False, "transformation": "Exponential decay weighted"},
    {"id": "f10", "name": "Analyst Surprise Sentiment Divergence",  "pipeline": "fp4", "factor_type": "Sentiment",    "coverage_universe": "US Equities",       "lookback_days": 30,  "is_proprietary": True,  "transformation": "Cross-signal divergence"},
    # fp5 → Macro Factor Builder
    {"id": "f11", "name": "Growth Nowcast Surprise",                 "pipeline": "fp5", "factor_type": "Macro",         "coverage_universe": "Multi-Asset",       "lookback_days": 30,  "is_proprietary": False, "transformation": "Surprise vs consensus"},
    {"id": "f12", "name": "Inflation Regime Classifier",            "pipeline": "fp5", "factor_type": "Macro",         "coverage_universe": "Multi-Asset",       "lookback_days": 90,  "is_proprietary": True,  "transformation": "Regime detection HMM"},
    {"id": "f13", "name": "Central Bank Hawkishness Score",         "pipeline": "fp5", "factor_type": "Macro",         "coverage_universe": "Fixed Income",      "lookback_days": 30,  "is_proprietary": False, "transformation": "NLP on CB statements"},
    # fp6 → Microstructure Flow Signals
    {"id": "f14", "name": "Options Implied Directional Bias",       "pipeline": "fp6", "factor_type": "Microstructure","coverage_universe": "US Equities",       "lookback_days": 5,   "is_proprietary": True,  "transformation": "Put/call skew adjusted"},
    {"id": "f15", "name": "Dark Pool Absorption Ratio",             "pipeline": "fp6", "factor_type": "Microstructure","coverage_universe": "US Equities",       "lookback_days": 3,   "is_proprietary": True,  "transformation": "Volume ratio"},
    # fp7 → ESG Risk Overlay
    {"id": "f16", "name": "ESG Controversy Momentum",               "pipeline": "fp7", "factor_type": "ESG",           "coverage_universe": "Global Equities",   "lookback_days": 180, "is_proprietary": False, "transformation": "Event-driven z-score"},
    {"id": "f17", "name": "Carbon Risk Adjusted Return Factor",     "pipeline": "fp7", "factor_type": "ESG",           "coverage_universe": "Global Equities",   "lookback_days": 365, "is_proprietary": False, "transformation": "Carbon-adjusted return"},
    # fp8 → Workforce Intelligence Signal
    {"id": "f18", "name": "Hiring Acceleration Signal",             "pipeline": "fp8", "factor_type": "Alternative",  "coverage_universe": "US Equities",       "lookback_days": 60,  "is_proprietary": False, "transformation": "YoY posting growth rate"},
    {"id": "f19", "name": "Executive Turnover Risk Score",          "pipeline": "fp8", "factor_type": "Alternative",  "coverage_universe": "US Equities",       "lookback_days": 90,  "is_proprietary": False, "transformation": "Abnormal LinkedIn activity"},
]

q_feat = """
UNWIND $features AS f
MATCH (p:FeaturePipeline {id: f.pipeline})
MERGE (n:Feature {id: f.id})
SET n += {name: f.name, factor_type: f.factor_type,
          coverage_universe: f.coverage_universe,
          lookback_days: f.lookback_days, is_proprietary: f.is_proprietary}
MERGE (p)-[r:GENERATES]->(n)
SET r.transformation = f.transformation
"""

with driver.session() as session:
    session.run(q_feat, features=features)
print(f"✅ {len(features)} features loaded")

## 7. Load Models

In [ ]:
models = [
    {"id": "m1", "name": "AlphaBlend US Equity",        "model_type": "Ensemble",          "asset_class": "US Equities",       "backtest_sharpe": 2.1,  "live_sharpe": 1.7,  "inception_year": 2019, "status": "Live"},
    {"id": "m2", "name": "Global Macro Rotator",        "model_type": "Linear Factor",    "asset_class": "Multi-Asset",       "backtest_sharpe": 1.6,  "live_sharpe": 1.3,  "inception_year": 2018, "status": "Live"},
    {"id": "m3", "name": "StatArb European Equities",   "model_type": "GBM",              "asset_class": "European Equities", "backtest_sharpe": 1.9,  "live_sharpe": 1.5,  "inception_year": 2020, "status": "Live"},
    {"id": "m4", "name": "Sentiment Momentum L/S",      "model_type": "LSTM",             "asset_class": "Global Equities",   "backtest_sharpe": 1.4,  "live_sharpe": 0.8,  "inception_year": 2021, "status": "Live"},
    {"id": "m5", "name": "Execution Timing Optimizer",  "model_type": "Signal Combiner", "asset_class": "US Equities",       "backtest_sharpe": 2.4,  "live_sharpe": 2.1,  "inception_year": 2022, "status": "Live"},
    {"id": "m6", "name": "ESG Tilt Factor Model",       "model_type": "Linear Factor",    "asset_class": "Global Equities",   "backtest_sharpe": 0.9,  "live_sharpe": 0.5,  "inception_year": 2022, "status": "Research"},
]

# Feature → Model with Shapley values (per model, shapley_values should sum to ~1)
feature_model_links = [
    # m1 — AlphaBlend US Equity
    {"f": "f1",  "m": "m1", "shapley_value": 0.22, "importance_rank": 1, "in_production": True},
    {"f": "f5",  "m": "m1", "shapley_value": 0.19, "importance_rank": 2, "in_production": True},
    {"f": "f6",  "m": "m1", "shapley_value": 0.17, "importance_rank": 3, "in_production": True},
    {"f": "f8",  "m": "m1", "shapley_value": 0.14, "importance_rank": 4, "in_production": True},
    {"f": "f10", "m": "m1", "shapley_value": 0.12, "importance_rank": 5, "in_production": True},
    {"f": "f18", "m": "m1", "shapley_value": 0.09, "importance_rank": 6, "in_production": True},
    {"f": "f19", "m": "m1", "shapley_value": 0.07, "importance_rank": 7, "in_production": False},
    # m2 — Global Macro Rotator
    {"f": "f11", "m": "m2", "shapley_value": 0.31, "importance_rank": 1, "in_production": True},
    {"f": "f12", "m": "m2", "shapley_value": 0.25, "importance_rank": 2, "in_production": True},
    {"f": "f3",  "m": "m2", "shapley_value": 0.20, "importance_rank": 3, "in_production": True},
    {"f": "f4",  "m": "m2", "shapley_value": 0.14, "importance_rank": 4, "in_production": True},
    {"f": "f13", "m": "m2", "shapley_value": 0.10, "importance_rank": 5, "in_production": True},
    # m3 — StatArb European Equities
    {"f": "f7",  "m": "m3", "shapley_value": 0.28, "importance_rank": 1, "in_production": True},
    {"f": "f4",  "m": "m3", "shapley_value": 0.22, "importance_rank": 2, "in_production": True},
    {"f": "f8",  "m": "m3", "shapley_value": 0.18, "importance_rank": 3, "in_production": True},
    {"f": "f12", "m": "m3", "shapley_value": 0.16, "importance_rank": 4, "in_production": True},
    {"f": "f9",  "m": "m3", "shapley_value": 0.10, "importance_rank": 5, "in_production": True},
    {"f": "f17", "m": "m3", "shapley_value": 0.06, "importance_rank": 6, "in_production": False},
    # m4 — Sentiment Momentum L/S
    {"f": "f8",  "m": "m4", "shapley_value": 0.35, "importance_rank": 1, "in_production": True},
    {"f": "f9",  "m": "m4", "shapley_value": 0.28, "importance_rank": 2, "in_production": True},
    {"f": "f10", "m": "m4", "shapley_value": 0.22, "importance_rank": 3, "in_production": True},
    {"f": "f2",  "m": "m4", "shapley_value": 0.15, "importance_rank": 4, "in_production": True},
    # m5 — Execution Timing Optimizer
    {"f": "f14", "m": "m5", "shapley_value": 0.40, "importance_rank": 1, "in_production": True},
    {"f": "f15", "m": "m5", "shapley_value": 0.35, "importance_rank": 2, "in_production": True},
    {"f": "f9",  "m": "m5", "shapley_value": 0.15, "importance_rank": 3, "in_production": True},
    {"f": "f1",  "m": "m5", "shapley_value": 0.10, "importance_rank": 4, "in_production": True},
    # m6 — ESG Tilt Factor Model (Research)
    {"f": "f16", "m": "m6", "shapley_value": 0.45, "importance_rank": 1, "in_production": True},
    {"f": "f17", "m": "m6", "shapley_value": 0.35, "importance_rank": 2, "in_production": True},
    {"f": "f11", "m": "m6", "shapley_value": 0.20, "importance_rank": 3, "in_production": False},
]

q_models = """
UNWIND $models AS m
MERGE (n:Model {id: m.id})
SET n += m
"""

q_used_in = """
UNWIND $links AS l
MATCH (f:Feature {id: l.f})
MATCH (m:Model   {id: l.m})
MERGE (f)-[r:USED_IN]->(m)
SET r.shapley_value   = l.shapley_value,
    r.importance_rank = l.importance_rank,
    r.in_production   = l.in_production
"""

with driver.session() as session:
    session.run(q_models,  models=models)
    session.run(q_used_in, links=feature_model_links)
print(f"✅ {len(models)} models and {len(feature_model_links)} USED_IN relationships loaded")

## 8. Load Strategies and PnL

In [ ]:
strategies = [
    {"id": "s1", "name": "Meridian US Equity L/S",        "aum_usd_m": 1200, "asset_class": "US Equities",       "approach": "Long/Short Equity",      "target_sharpe": 1.5, "inception_year": 2019},
    {"id": "s2", "name": "Atlas Global Macro",            "aum_usd_m": 850,  "asset_class": "Multi-Asset",       "approach": "Global Macro",           "target_sharpe": 1.2, "inception_year": 2018},
    {"id": "s3", "name": "Vega European StatArb",         "aum_usd_m": 650,  "asset_class": "European Equities", "approach": "Statistical Arbitrage", "target_sharpe": 1.6, "inception_year": 2020},
    {"id": "s4", "name": "Helix Execution Alpha",         "aum_usd_m": 400,  "asset_class": "US Equities",       "approach": "Execution Alpha",        "target_sharpe": 2.0, "inception_year": 2022},
]

model_strategy_links = [
    {"m": "m1", "s": "s1", "signal_weight": 0.55, "since_year": 2019},
    {"m": "m4", "s": "s1", "signal_weight": 0.30, "since_year": 2021},
    {"m": "m6", "s": "s1", "signal_weight": 0.15, "since_year": 2022},
    {"m": "m2", "s": "s2", "signal_weight": 0.70, "since_year": 2018},
    {"m": "m4", "s": "s2", "signal_weight": 0.30, "since_year": 2021},
    {"m": "m3", "s": "s3", "signal_weight": 0.80, "since_year": 2020},
    {"m": "m2", "s": "s3", "signal_weight": 0.20, "since_year": 2021},
    {"m": "m5", "s": "s4", "signal_weight": 1.00, "since_year": 2022},
]

pnl_records = [
    # s1 — Meridian US Equity L/S
    {"id": "p1a", "strategy": "s1", "period": "2022-FY", "gross_pnl_usd_m": 94.0,  "net_pnl_usd_m": 78.0,  "sharpe": 1.5, "max_drawdown_pct": 8.2},
    {"id": "p1b", "strategy": "s1", "period": "2023-FY", "gross_pnl_usd_m": 142.0, "net_pnl_usd_m": 121.0, "sharpe": 1.8, "max_drawdown_pct": 6.5},
    {"id": "p1c", "strategy": "s1", "period": "2024-FY", "gross_pnl_usd_m": 168.0, "net_pnl_usd_m": 144.0, "sharpe": 2.0, "max_drawdown_pct": 5.1},
    # s2 — Atlas Global Macro
    {"id": "p2a", "strategy": "s2", "period": "2022-FY", "gross_pnl_usd_m": 62.0,  "net_pnl_usd_m": 51.0,  "sharpe": 1.1, "max_drawdown_pct": 11.0},
    {"id": "p2b", "strategy": "s2", "period": "2023-FY", "gross_pnl_usd_m": 75.0,  "net_pnl_usd_m": 62.0,  "sharpe": 1.2, "max_drawdown_pct": 9.8},
    {"id": "p2c", "strategy": "s2", "period": "2024-FY", "gross_pnl_usd_m": 91.0,  "net_pnl_usd_m": 78.0,  "sharpe": 1.4, "max_drawdown_pct": 8.4},
    # s3 — Vega European StatArb
    {"id": "p3a", "strategy": "s3", "period": "2022-FY", "gross_pnl_usd_m": 55.0,  "net_pnl_usd_m": 46.0,  "sharpe": 1.3, "max_drawdown_pct": 7.5},
    {"id": "p3b", "strategy": "s3", "period": "2023-FY", "gross_pnl_usd_m": 72.0,  "net_pnl_usd_m": 61.0,  "sharpe": 1.6, "max_drawdown_pct": 6.2},
    {"id": "p3c", "strategy": "s3", "period": "2024-FY", "gross_pnl_usd_m": 88.0,  "net_pnl_usd_m": 75.0,  "sharpe": 1.8, "max_drawdown_pct": 5.8},
    # s4 — Helix Execution Alpha
    {"id": "p4a", "strategy": "s4", "period": "2023-FY", "gross_pnl_usd_m": 48.0,  "net_pnl_usd_m": 42.0,  "sharpe": 2.0, "max_drawdown_pct": 3.2},
    {"id": "p4b", "strategy": "s4", "period": "2024-FY", "gross_pnl_usd_m": 71.0,  "net_pnl_usd_m": 63.0,  "sharpe": 2.3, "max_drawdown_pct": 2.8},
]

q_strat = """
UNWIND $strategies AS s
MERGE (n:Strategy {id: s.id})
SET n += s
"""

q_powers = """
UNWIND $links AS l
MATCH (m:Model    {id: l.m})
MATCH (s:Strategy {id: l.s})
MERGE (m)-[r:POWERS]->(s)
SET r.signal_weight = l.signal_weight,
    r.since_year    = l.since_year
"""

q_pnl = """
UNWIND $records AS r
MATCH (s:Strategy {id: r.strategy})
MERGE (p:PnL {id: r.id})
SET p += {period: r.period, gross_pnl_usd_m: r.gross_pnl_usd_m,
          net_pnl_usd_m: r.net_pnl_usd_m, sharpe: r.sharpe,
          max_drawdown_pct: r.max_drawdown_pct}
MERGE (s)-[:PRODUCES]->(p)
"""

with driver.session() as session:
    session.run(q_strat,  strategies=strategies)
    session.run(q_powers, links=model_strategy_links)
    session.run(q_pnl,    records=pnl_records)
print(f"✅ {len(strategies)} strategies, {len(model_strategy_links)} POWERS rels, {len(pnl_records)} PnL records loaded")

## 9. Load Substitutability & Feature Correlation Relationships

In [ ]:
substitutes = [
    # Credit card datasets overlap — TransactIQ US could partially sub for parking lot satellite
    {"d1": "d3",  "d2": "d1",  "overlap_score": 0.45, "cost_delta_usd":  270000},
    # Two sentiment datasets from same vendor overlap heavily
    {"d1": "d5",  "d2": "d6",  "overlap_score": 0.60, "cost_delta_usd":   50000},
    # ESG datasets from GreenSignal are partially substitutable
    {"d1": "d11", "d2": "d12", "overlap_score": 0.50, "cost_delta_usd":   65000},
    # Workforce datasets overlap moderately
    {"d1": "d13", "d2": "d14", "overlap_score": 0.35, "cost_delta_usd":   25000},
    # Options flow and dark pool data are complementary but partially redundant
    {"d1": "d9",  "d2": "d10", "overlap_score": 0.40, "cost_delta_usd":   90000},
]

feature_correlations = [
    # Sentiment features are correlated
    {"f1": "f8",  "f2": "f9",  "r_squared": 0.61, "detected_date": "2024-03-15"},
    {"f1": "f8",  "f2": "f10", "r_squared": 0.53, "detected_date": "2024-03-15"},
    # Consumer spend and retail footfall are correlated
    {"f1": "f1",  "f2": "f5",  "r_squared": 0.48, "detected_date": "2023-11-01"},
    # Dark pool and options flow are correlated
    {"f1": "f14", "f2": "f15", "r_squared": 0.57, "detected_date": "2024-06-20"},
    # ESG factors are correlated
    {"f1": "f16", "f2": "f17", "r_squared": 0.44, "detected_date": "2024-01-10"},
    # Macro features
    {"f1": "f11", "f2": "f12", "r_squared": 0.38, "detected_date": "2023-09-05"},
]

q_subs = """
UNWIND $subs AS s
MATCH (a:Dataset {id: s.d1})
MATCH (b:Dataset {id: s.d2})
MERGE (a)-[r:SUBSTITUTABLE_FOR]->(b)
SET r.overlap_score    = s.overlap_score,
    r.cost_delta_usd   = s.cost_delta_usd
"""

q_corr = """
UNWIND $corrs AS c
MATCH (a:Feature {id: c.f1})
MATCH (b:Feature {id: c.f2})
MERGE (a)-[r:CORRELATED_WITH]->(b)
SET r.r_squared     = c.r_squared,
    r.detected_date = c.detected_date
"""

with driver.session() as session:
    session.run(q_subs, subs=substitutes)
    session.run(q_corr, corrs=feature_correlations)
print(f"✅ {len(substitutes)} SUBSTITUTABLE_FOR and {len(feature_correlations)} CORRELATED_WITH relationships loaded")

## 10. Verify the Graph

In [ ]:
import pandas as pd

summary_query = """
MATCH (n)
RETURN labels(n)[0] AS label, count(n) AS count
ORDER BY count DESC
"""

rel_query = """
MATCH ()-[r]->()
RETURN type(r) AS relationship, count(r) AS count
ORDER BY count DESC
"""

with driver.session() as session:
    node_results = session.run(summary_query).data()
    rel_results  = session.run(rel_query).data()

print("=== NODE COUNTS ===")
print(pd.DataFrame(node_results).to_string(index=False))
print("\n=== RELATIONSHIP COUNTS ===")
print(pd.DataFrame(rel_results).to_string(index=False))

In [ ]:
# Quick sanity check — trace one full path from vendor to PnL
path_query = """
MATCH path = (v:DataVendor {name: 'TransactIQ'})
              -[:PROVIDES]->(d:Dataset)
              -[:FEEDS]->(fp:FeaturePipeline)
              -[:GENERATES]->(f:Feature)
              -[:USED_IN]->(m:Model)
              -[:POWERS]->(s:Strategy)
              -[:PRODUCES]->(p:PnL)
RETURN v.name AS vendor, d.name AS dataset, fp.name AS pipeline,
       f.name AS feature, m.name AS model, s.name AS strategy,
       p.period AS period, p.net_pnl_usd_m AS net_pnl_m
ORDER BY p.period
LIMIT 10
"""

with driver.session() as session:
    results = session.run(path_query).data()

print("=== SAMPLE PATH: TransactIQ → PnL ===")
pd.set_option('display.max_colwidth', 40)
print(pd.DataFrame(results).to_string(index=False))

---
## ✅ Graph loaded successfully!

You can now:
- Open **Neo4j Bloom** in the Aura Console to explore the graph visually
- Run the Cypher queries in the `queries/` folder for each analytical lens
- Follow the `bloom/BLOOM_GUIDE.md` to configure search phrases and scene actions